# Práctica 5 — Red neuronal densa con Fashion-MNIST (prendas de vestir)

**Objetivo de esta práctica:** reutilizar la misma red de la práctica 4, pero ahora para clasificar imágenes de **prendas de vestir** en 10 categorías (camiseta, pantalón, zapatilla, etc.).

Fashion-MNIST tiene el mismo formato que MNIST (imágenes de 28 × 28 en escala de grises), por eso el código es casi igual. La diferencia es que aquí las etiquetas se traducen a nombres de prendas, se revisan varias predicciones a la vez y se mide la precisión global con `evaluate`.

## 1. Importar librerías

In [ ]:
import tensorflow as tf                                # TensorFlow: para construir y entrenar la red neuronal
import matplotlib.pyplot as plt                        # Para mostrar las imágenes de las prendas
from tensorflow.keras.datasets import fashion_mnist    # Dataset Fashion-MNIST: prendas de vestir (70 000 imágenes)

## 2. Cargar el dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

## 3. Explorar la forma de los datos

In [ ]:
print("Forma de la imagen:", x_test[0].shape)
print("Forma de x_train:", x_train.shape)
print("Forma de x_test:", x_test.shape)
print("Total de muestras:", (len(y_train) + len(y_test)))

## 4. Visualizar las primeras 10 imágenes de prueba con su etiqueta

In [ ]:
plt.figure(figsize=(10, 10))
for s in range(10):
    plt.subplot(2, 5, s + 1)
    plt.imshow(x_test[s], cmap="gray")
    plt.title(y_test[s])                      # Por ahora la etiqueta es solo un número (0 al 9)
plt.tight_layout()

## 5. Normalizar los píxeles (0 a 1)

In [ ]:
x_train = x_train / 255.0
x_test = x_test / 255.0

## 6. Construir el modelo

Misma arquitectura que en MNIST: `Flatten` → `Dense(128, relu)` → `Dense(10, softmax)`.

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),        # Convierte la imagen 2D en un vector plano
    tf.keras.layers.Dense(128, activation="relu"),        # Capa oculta: 128 neuronas
    tf.keras.layers.Dense(10, activation="softmax")       # Capa de salida: 10 probabilidades (una por prenda)
])

## 7. Compilar y entrenar (15 épocas)

In [ ]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=15)

## 8. Nombres de las clases

Las etiquetas del dataset son números del 0 al 9. Esta lista permite traducir cada número al nombre de la prenda.

In [ ]:
class_names = [
    "T-shirt/top (Camiseta)", "Trouser (Pantalón)", "Pullover (Pulóver)",
    "Dress (Vestido)", "Coat (Abrigo)", "Sandal (Sandalia)",
    "Shirt (Camisa)", "Sneaker (Zapatilla)", "Bag (Bolso)", "Ankle boot (Botín)"
]
print(class_names[0])

## 9. Predecir una prenda

In [ ]:
muestra = x_test[111].reshape(1, 28, 28)
predict = model.predict(muestra)

print("Salida de probabilidades de nuestro softmax:", predict)
print("Afinando la salida:", predict.argmax(1), class_names[predict.argmax(1)[0]])
print("Etiqueta real:", y_test[111], class_names[y_test[111]])
plt.imshow(muestra.reshape(28, 28), cmap="gray")

## 10. Predecir 10 prendas a la vez

Se toman las imágenes 111 a 120, se predicen todas juntas y se compara la predicción con la etiqueta real de cada una.

In [ ]:
muestras = x_test[111:121]
predicts = model.predict(muestras)
indices_predicciones = predicts.argmax(1)

for i in range(10):
    idx_real = y_test[111 + i]               # Índice real de cada prenda
    idx_pred = indices_predicciones[i]       # Índice predicho por el modelo

    print(f"--- Prenda {111 + i} ---")
    print("Afinando la salida:", idx_pred, class_names[idx_pred])
    print("Etiqueta real:", idx_real, class_names[idx_real])
    print("-" * 30)

plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(muestras[i], cmap="gray")
    plt.title(f"P: {class_names[indices_predicciones[i]].split(' ')[0]}")   # P: prenda predicha
    plt.axis("off")
plt.tight_layout()
plt.show()

## 11. Precisión global con los datos de prueba

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"El modelo tiene una precisión global del: {test_acc * 100:.2f}%")

## 🗒️ Conclusión

La misma red que reconoce dígitos sirve para clasificar prendas, aunque con menor precisión, porque algunas prendas se parecen mucho entre sí (camisa, camiseta y pulóver). `evaluate` permite medir el desempeño con datos que la red nunca vio durante el entrenamiento.